In [1]:
pip install transformers torch peft librosa accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from peft import PeftModel
import librosa

def transcribe_punjabi_audio(audio_path, repo_id, base_model_id="openai/whisper-large-v3-turbo"):
    """
    Transcribes a local audio file using a fine-tuned Whisper model with LoRA adapters.

    Args:
        audio_path (str): Path to the local audio file (e.g., .mp3, .wav, .m4a).
        repo_id (str): Your Hugging Face repository ID containing the LoRA adapters.
        base_model_id (str): The underlying base architecture model.
    """
    # 1. Determine device setup
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    print(f"Loading processor and base model: {base_model_id}...")

    # 2. Load the processor (combines feature extractor and tokenizer)
    # Using your fine-tuned repo if it contains the updated processor config, else fallback to base
    try:
        processor = AutoProcessor.from_pretrained(repo_id, language="punjabi", task="transcribe")
    except Exception:
        processor = AutoProcessor.from_pretrained(base_model_id, language="punjabi", task="transcribe")

    # 3. Load base model with low-resource settings
    # (Matches your notebook's requirement to run on limited GPU memory like a T4)
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        base_model_id,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        device_map="auto" if device == "cuda" else None
    )

    print(f"Merging trained LoRA adapters from {repo_id}...")
    # 4. Wrap the base model with your custom LoRA adapter layers
    model = PeftModel.from_pretrained(base_model, repo_id)
    model.to(device)
    model.eval()

    print(f"Processing audio file: {audio_path}...")
    # 5. Load and resample the audio to 16,000 Hz (Crucial for Whisper)
    audio_array, sampling_rate = librosa.load(audio_path, sr=16000)

    # 6. Extract log-mel spectrogram features from the raw audio waveform
    input_features = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to(device).to(torch_dtype)

    print("Generating transcription...")
    # 7. Predict token IDs
    with torch.no_grad():
        # Force the decoder to use Punjabi language constraints
        predicted_ids = model.generate(
            input_features,
            language="punjabi",
            task="transcribe"
        )

    # 8. Decode the token IDs back into readable Gurmukhi script text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    return transcription

# --- Execution ---
if __name__ == "__main__":
    # Replace with the path to your target audio file
    TARGET_AUDIO = "/content/Full Narration_MarauliKhurad.m4a"

    # Your Hugging Face repository ID containing the weights uploaded in your notebook
    HF_REPO_ID = "Garden2006/whisper-large-v3-turbo-gurmukhi-lora"

    try:
        result = transcribe_punjabi_audio(audio_path=TARGET_AUDIO, repo_id=HF_REPO_ID)
        print("\n--- Transcription Result ---")
        print(result)
    except FileNotFoundError:
        print(f"Error: Could not find audio file at '{TARGET_AUDIO}'. Please update the path variable.")

Loading processor and base model: openai/whisper-large-v3-turbo...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/410 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

Merging trained LoRA adapters from Garden2006/whisper-large-v3-turbo-gurmukhi-lora...


adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [3]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [4]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from peft import PeftModel
import librosa

def transcribe_punjabi_audio(audio_path, repo_id, base_model_id="openai/whisper-large-v3-turbo"):
    """
    Transcribes a local audio file using a fine-tuned Whisper model with LoRA adapters.

    Args:
        audio_path (str): Path to the local audio file (e.g., .mp3, .wav, .m4a).
        repo_id (str): Your Hugging Face repository ID containing the LoRA adapters.
        base_model_id (str): The underlying base architecture model.
    """
    # 1. Determine device setup
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    print(f"Loading processor and base model: {base_model_id}...")

    # 2. Load the processor (combines feature extractor and tokenizer)
    # Using your fine-tuned repo if it contains the updated processor config, else fallback to base
    try:
        processor = AutoProcessor.from_pretrained(repo_id, language="punjabi", task="transcribe")
    except Exception:
        processor = AutoProcessor.from_pretrained(base_model_id, language="punjabi", task="transcribe")

    # 3. Load base model with low-resource settings
    # (Matches your notebook's requirement to run on limited GPU memory like a T4)
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        base_model_id,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        device_map="auto" if device == "cuda" else None
    )

    print(f"Merging trained LoRA adapters from {repo_id}...")
    # 4. Wrap the base model with your custom LoRA adapter layers
    model = PeftModel.from_pretrained(base_model, repo_id)
    model.to(device)
    model.eval()

    print(f"Processing audio file: {audio_path}...")
    # 5. Load and resample the audio to 16,000 Hz (Crucial for Whisper)
    audio_array, sampling_rate = librosa.load(audio_path, sr=16000)

    # 6. Extract log-mel spectrogram features from the raw audio waveform
    input_features = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to(device).to(torch_dtype)

    print("Generating transcription...")
    # 7. Predict token IDs
    with torch.no_grad():
        # Force the decoder to use Punjabi language constraints
        predicted_ids = model.generate(
            input_features,
            language="punjabi",
            task="transcribe"
        )

    # 8. Decode the token IDs back into readable Gurmukhi script text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    return transcription

# --- Execution ---
if __name__ == "__main__":
    # Replace with the path to your target audio file
    TARGET_AUDIO = "/content/Full Narration_MarauliKhurad.m4a"

    # Your Hugging Face repository ID containing the weights uploaded in your notebook
    HF_REPO_ID = "Garden2006/whisper-large-v3-turbo-gurmukhi-lora"

    try:
        result = transcribe_punjabi_audio(audio_path=TARGET_AUDIO, repo_id=HF_REPO_ID)
        print("\n--- Transcription Result ---")
        print(result)
    except FileNotFoundError:
        print(f"Error: Could not find audio file at '{TARGET_AUDIO}'. Please update the path variable.")

Loading processor and base model: openai/whisper-large-v3-turbo...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Merging trained LoRA adapters from Garden2006/whisper-large-v3-turbo-gurmukhi-lora...


adapter_model.safetensors:   0%|          | 0.00/26.2M [00:00<?, ?B/s]

Processing audio file: /content/Full Narration_MarauliKhurad.m4a...


/tmp/ipykernel_2423/4237176213.py:45: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generating transcription...


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[


--- Transcription Result ---
ਸਾਹ ਸ਼ਿਕਾਲਜੀ ਮੀਟਿੰ ਸ਼ੜਿੂਲ ਬਿਲਜ ਮੁਡੋਲੀ ਖੁਰਦ ਡੇਟ ਦੋ ਅਪਰਾਈਲ ਦੋ ਇਆਰ ਛੱਬੀ ਬਿਲਜ ਮਡੋਲੀ ਖੁਰਦ ਪੰਚਿਥ ਮਡੋਲੀ ਖੁਰਦ ਬਲੋਕ ਮਰੰਡਾ ਡੇਸਟੀ ਕਰੂਪ ਨਗਰ ਨੇ ਮੌਫਦਾ ਕੋਡੀਨੇਟਰ ਅਮਰਿਤ ਪ੍ਰਿਤ ਸਿੰਘਾਨਿਜ਼ ਸ਼ੀਰੀਮਤੀ ਨਮੀਸ਼ਾ ਮਹਾਜ਼ਣਜੀ ਇਵਿਂਟ ਸਟਾਟ ਟਾਈਮ ਗਿਆਾਰ ਦਿਨ ਬੀਰਬਾਰ


In [5]:
if __name__ == "__main__":
    TARGET_AUDIO = "/content/Full Narration_MarauliKhurad.m4a"
    HF_REPO_ID = "Garden2006/whisper-large-v3-turbo-gurmukhi-lora"

    # 1. Run the original function
    result = transcribe_punjabi_audio(audio_path=TARGET_AUDIO, repo_id=HF_REPO_ID)

    # 2. Print it to the screen like before
    print("\n--- Transcription Result ---")
    print(result)

    # 3. ADD THIS: Save it to a file in your Colab sidebar
    with open("short_transcription.txt", "w", encoding="utf-8") as f:
        f.write(result)
    print("\n💾 Saved to Colab files as 'short_transcription.txt'!")

Loading processor and base model: openai/whisper-large-v3-turbo...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Merging trained LoRA adapters from Garden2006/whisper-large-v3-turbo-gurmukhi-lora...
Processing audio file: /content/Full Narration_MarauliKhurad.m4a...


/tmp/ipykernel_2423/4237176213.py:45: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Generating transcription...

--- Transcription Result ---
ਸਾਹ ਸ਼ਿਕਾਲਜੀ ਮੀਟਿੰ ਸ਼ੜਿੂਲ ਬਿਲਜ ਮੁਡੋਲੀ ਖੁਰਦ ਡੇਟ ਦੋ ਅਪਰਾਈਲ ਦੋ ਇਆਰ ਛੱਬੀ ਬਿਲਜ ਮਡੋਲੀ ਖੁਰਦ ਪੰਚਿਥ ਮਡੋਲੀ ਖੁਰਦ ਬਲੋਕ ਮਰੰਡਾ ਡੇਸਟੀ ਕਰੂਪ ਨਗਰ ਨੇ ਮੌਫਦਾ ਕੋਡੀਨੇਟਰ ਅਮਰਿਤ ਪ੍ਰਿਤ ਸਿੰਘਾਨਿਜ਼ ਸ਼ੀਰੀਮਤੀ ਨਮੀਸ਼ਾ ਮਹਾਜ਼ਣਜੀ ਇਵਿਂਟ ਸਟਾਟ ਟਾਈਮ ਗਿਆਾਰ ਦਿਨ ਬੀਰਬਾਰ

💾 Saved to Colab files as 'short_transcription.txt'!
